# Ionosphere S2CNN — SFNO Training (Reformatted Data)

Spherical Fourier Neural Operator for global ionospheric dTEC forecasting. **Training only** —
no baselines, no eval, no leftover fake-data code. `ionosphere_baselines.ipynb` and
`ionosphere_eval.ipynb` are built separately once this checkpoint exists.

**Data:** Hayden's reformatted delivery (`new_data/`), fixing two bugs in the previous data:
IRI baseline computed in MST instead of UTC (shifted diurnal alignment), and a 2014 IONEX
resolution change (2hr -> 1hr) that gave pre-/post-2014 windows different real lookback spans
under the same tensor shape.

**Consequences of the reformat, baked into this notebook (see README cell below for the source):**
- Horizons are now **t+2h / t+4h / t+6h**, not t+1h/t+2h/t+3h — derived dynamically from
  `metadata.json`'s `cadence_seconds`, not hardcoded, so a future cadence change won't silently
  mislabel results again.
- Latitude ordering is checked against what `torch-harmonics` expects at runtime (Cell 3) rather
  than assumed, and flipped if needed — the data's own convention (ascending south->north) is
  the opposite of IONEX's native ordering per Hayden's note, and may or may not also oppose
  `torch-harmonics`'s internal convention.
- Normalization constants (`tec_mean`, `tec_std`) are read from `metadata.json` at runtime, never
  hardcoded — already true before this reformat, still true now, just confirming it survives.
- No assumption that train + val + test window counts sum to a "total" — 26 windows are purged/
  embargoed at split boundaries.

**Removed vs. earlier notebooks:** `FakeTECDataset`, all baseline classes (Persistence,
Climatology, OMNI-only GRU, Flat CNN), `evaluate_baselines`, the dangling `make_fake_climatology`
call, the old `falisha_windows_gl23x45.tar.gz` path. Absolute-TEC (climatology-added-back) scoring
is deliberately **not** in this notebook — climatology still isn't delivered for this data, and
that scoring will live in the eval notebook or a 4th post-processing script, not here.

**Checkpoint produced:** `best_model.pt`, backed up to `/content/drive/MyDrive/tec_data/`.

## ⚠️ TEMPORARY — Hayden's handover notes (delete this cell once internalized/acted on)

Reproduced verbatim so nothing gets lost before it's fully baked into the code below.

**Four things that will silently produce wrong results if assumed otherwise:**

1. Latitudes ascend south -> north (-84.14 ... +84.14). Row 0 is the southernmost band — the
   opposite of IONEX's native ordering. Plot against `lats.npy`, don't assume.
2. The horizon is +2/+4/+6 hours, not +1/+2/+3. Frames are 7200s apart; `input_steps: 6` and
   `target_steps: 3` are counts, and only `cadence_seconds` makes them a physical horizon.
3. Everything is normalized, targets included. To get predictions back in TECU:
   `pred * tec_std + tec_mean` using the constants in `metadata.json` (3.6707 / 7.5342). Those
   come from the training split only.
4. The splits do not partition the windows. 26 were purged or embargoed at boundaries. Don't
   expect the three counts to sum to a "total windows" figure.

**Two caveats worth stating in the handover:**

- Train and test differ in solar activity — F10.7 mean 108.3 vs 167.5, ratio 0.65, with test p95
  (237.6) above train p95 (193.7). The model will be evaluated somewhat outside its training
  range. Report test metrics stratified by activity band rather than as a single number.
- `val` ends 2022-11-26, not year-end. That's a real 35-day gap in the CODE archive, not a bug.

*(Items 1-2 are handled in Cells 2-3 below. Item 3 was already metadata-driven before this
reformat. Item 4 just means: don't write an assertion that counts sum to a total. The solar-
activity stratification and absolute-TEC-in-TECU reporting happen in the eval notebook, not
here — this notebook only trains and checkpoints.)*

## Cell 1 — Setup: mount Drive, copy data locally, install, imports, GPU check

All in one cell. `new_data/` is a live folder in Drive (not a tarball) — copied locally first since repeated epoch reads over the Drive FUSE mount are slow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, json, warnings
from pathlib import Path

DRIVE_DATA_DIR = "/content/drive/MyDrive/tec_data/new_data"
LOCAL_DATA_DIR = "new_data"

# Copy only real data files (.npy / .json) -- Hayden's Drive folder also contains
# README.md.gdoc, a Google Docs shortcut (not a real file) that shutil.copytree can't
# read through the Drive FUSE mount and isn't part of the dataset anyway.
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
for fname in os.listdir(DRIVE_DATA_DIR):
    if fname.endswith('.npy') or fname.endswith('.json'):
        dst = os.path.join(LOCAL_DATA_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy(os.path.join(DRIVE_DATA_DIR, fname), dst)
print("Local data files:", sorted(os.listdir(LOCAL_DATA_DIR)))

!pip install -q torch-harmonics wandb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import wandb

from torch_harmonics import RealSHT, InverseRealSHT
from torch_harmonics.quadrature import legendre_gauss_weights

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: No GPU detected. Runtime -> Change runtime type -> T4 GPU')


Mounted at /content/drive
Local data files: ['lats.npy', 'lons.npy', 'metadata.json', 'test_omni_input.npy', 'test_target.npy', 'test_tec_input.npy', 'test_window_start_times.npy', 'train_omni_input.npy', 'train_target.npy', 'train_tec_input.npy', 'train_window_start_times.npy', 'val_omni_input.npy', 'val_target.npy', 'val_tec_input.npy', 'val_window_start_times.npy']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.6/537.6 kB 12.4 MB/s eta 0:00:00
Using device: cuda


## Cell 2 — Config

Grid size still fixed (H/W/l_max from the resolved project convention), everything data-derived (normalization constants, horizon names) is read from `metadata.json` rather than hardcoded.

In [ ]:
with open(f"{LOCAL_DATA_DIR}/metadata.json") as f:
    _metadata = json.load(f)

_cadence_hours = _metadata['cadence_seconds'] / 3600
_n_horizons    = _metadata['target_steps']
HORIZON_NAMES  = [f"t+{int(_cadence_hours * (i + 1))}h" for i in range(_n_horizons)]

CONFIG = {
    # Grid -- H/W from metadata directly. l_max/m_max: metadata's "lmax" (22) is the raw
    # degree L_max; torch-harmonics' lmax kwarg wants a *count* (degrees 0..L_max), i.e.
    # L_max + 1 = 23 -- this exact conversion was already resolved earlier in the project,
    # derived here from metadata rather than a bare hardcoded 23 so it stays traceable.
    'H':     _metadata['nlat'],
    'W':     _metadata['nlon'],
    'l_max': _metadata['lmax'] + 1,
    'm_max': _metadata['lmax'] + 1,

    # Data (derived from metadata.json, not hardcoded)
    'n_timesteps':     _metadata['input_steps'],
    'n_omni_features': len(_metadata['omni_features']),
    'n_horizons':      _n_horizons,
    'horizon_names':   HORIZON_NAMES,
    'tec_mean':        _metadata['normalization']['tec_mean'],
    'tec_std':         _metadata['normalization']['tec_std'],
    'kp_feature_idx':  _metadata['omni_features'].index('kp_3hour'),
    'data_root':       LOCAL_DATA_DIR,

    # Model
    'gru_hidden_size': 128,
    'sfno_channels':   [64, 128, 128, 64],
    'n_sfno_blocks':   4,

    # Training
    'batch_size':     64,
    'learning_rate':  1e-4,
    'n_epochs':       20,
    'lambda_sobolev': 1e-7,
    'n_kp_bins':      18,

    # W&B
    'wandb_project':  'tec-sfno',
    'wandb_run_name': 'residual-reformatted-data-run-1',
}

print(f"Grid: {CONFIG['H']}x{CONFIG['W']} | l_max(count)={CONFIG['l_max']} (metadata lmax={_metadata['lmax']})")
print(f"Horizons: {CONFIG['horizon_names']}  (cadence={_metadata['cadence_seconds']}s)")
print(f"tec_mean={CONFIG['tec_mean']}, tec_std={CONFIG['tec_std']}  (train split only)")
print(f"n_omni_features={CONFIG['n_omni_features']}, kp_feature_idx={CONFIG['kp_feature_idx']}")


Grid: 23x45 | l_max(count)=23 (metadata lmax=22)
Horizons: ['t+2h', 't+4h', 't+6h']  (cadence=7200s)
tec_mean=3.6707473956641246, tec_std=7.534159202445188  (train split only)
n_omni_features=6, kp_feature_idx=5


## Cell 3 — ⚠️ TEMPORARY: latitude orientation check (verify once, then delete)

Hayden's note says the data ascends south->north, opposite IONEX's native order — but doesn't say whether that also opposes `torch-harmonics`'s internal convention, which is the thing that actually matters for the SHT. This checks at runtime rather than assuming, and sets `CONFIG['flip_latitude']` accordingly. **Read the printed output once, confirm it makes sense, then this cell (and the flip logic reading `CONFIG['flip_latitude']` in Cell 4) can be deleted if you're confident — but don't delete before checking at least once.**

In [ ]:
cos_theta, _ = legendre_gauss_weights(CONFIG['H'])
# torch-harmonics convention: theta = colatitude, 0 at north pole -> lat = 90 - degrees(theta)
lat_gl = 90.0 - np.degrees(np.arccos(cos_theta.numpy()))

data_lats = np.load(f"{CONFIG['data_root']}/lats.npy")

print('torch-harmonics implied lat order (first/last 3):', lat_gl[:3].round(3), lat_gl[-3:].round(3))
print('data lat order            (first/last 3):', data_lats[:3].round(3), data_lats[-3:].round(3))

_data_ascending = data_lats[0] < data_lats[-1]
_gl_ascending   = lat_gl[0] < lat_gl[-1]
CONFIG['flip_latitude'] = bool(_data_ascending != _gl_ascending)

print(f"data ascending: {_data_ascending}  |  torch-harmonics ascending: {_gl_ascending}")
print(f"=> CONFIG['flip_latitude'] = {CONFIG['flip_latitude']}")

torch-harmonics implied lat order (first/last 3): [-84.137 -76.542 -68.903] [68.903 76.542 84.137]
data lat order            (first/last 3): [-84.137 -76.542 -68.903] [68.903 76.542 84.137]
data ascending: True  |  torch-harmonics ascending: True
=> CONFIG['flip_latitude'] = False


## Cell 4 — Dataset

`FalishaDTECDataset` inlined directly (rather than depending on a separately uploaded/fetched `falisha_dataset.py`) so an overnight run doesn't depend on a manual upload step succeeding first. Applies the latitude flip from Cell 3 if needed, and returns each sample's dataset index for future climatology alignment (unused here, kept for consistency with the eval notebook).

In [ ]:
class FalishaDTECDataset(Dataset):
    """Inlined from data_pull/falisha_dataset.py (Hayden), reformatted 'new_data/' layout."""
    def __init__(self, root, split, flip_latitude=False):
        self.root = Path(root)
        self.split = split
        self.flip_latitude = flip_latitude

        self.tec_input  = np.load(self.root / f"{split}_tec_input.npy", mmap_mode="r")
        self.omni_input = np.load(self.root / f"{split}_omni_input.npy", mmap_mode="r")
        self.target      = np.load(self.root / f"{split}_target.npy", mmap_mode="r")
        self.window_start_times = np.load(self.root / f"{split}_window_start_times.npy", mmap_mode="r")

    def __len__(self):
        return int(self.tec_input.shape[0])

    def __getitem__(self, idx):
        tec_input = np.array(self.tec_input[idx], dtype=np.float32, copy=True)
        target    = np.array(self.target[idx], dtype=np.float32, copy=True)
        if self.flip_latitude:
            tec_input = np.flip(tec_input, axis=-2).copy()
            target    = np.flip(target, axis=-2).copy()
        return {
            "tec_input":  torch.from_numpy(tec_input),
            "omni_input": torch.from_numpy(np.array(self.omni_input[idx], dtype=np.float32, copy=True)),
            "target":     torch.from_numpy(target),
            "timestamp":  torch.tensor(int(self.window_start_times[idx]), dtype=torch.int64),
            "index":      torch.tensor(idx, dtype=torch.long),
        }


def make_dataloader(config, split, shuffle=None):
    if shuffle is None:
        shuffle = (split == 'train')
    ds = FalishaDTECDataset(config['data_root'], split, flip_latitude=config['flip_latitude'])
    return DataLoader(ds, batch_size=config['batch_size'], shuffle=shuffle)


_loader = make_dataloader(CONFIG, split='train')
_batch  = next(iter(_loader))
print('tec_input:', _batch['tec_input'].shape, '| omni_input:', _batch['omni_input'].shape,
      '| target:', _batch['target'].shape)
print(f"train={len(_loader.dataset)}  (note: train+val+test won't sum to a clean total -- "
      f"26 windows purged/embargoed at split boundaries, per Hayden)")

tec_input: torch.Size([64, 6, 23, 45]) | omni_input: torch.Size([64, 6, 6]) | target: torch.Size([64, 3, 23, 45])
train=85087  (note: train+val+test won't sum to a clean total -- 26 windows purged/embargoed at split boundaries, per Hayden)


## Cell 5 — Precompute: GL area weights, Sobolev weights, storm weight table

In [ ]:
def make_gl_weights(H, device):
    _, w = legendre_gauss_weights(H)
    w    = w.to(torch.float32).to(device)
    return (w / w.sum()).view(1, 1, H, 1)


def make_sobolev_weights(l_max, device):
    l = torch.arange(l_max, dtype=torch.float32, device=device)
    return (1 + l * (l + 1)).view(1, 1, l_max, 1)


def build_storm_weight_table(kp_values, n_bins):
    kp_values = np.asarray(kp_values)
    bin_edges = np.linspace(kp_values.min(), kp_values.max(), n_bins + 1)
    counts, _ = np.histogram(kp_values, bins=bin_edges)
    density   = np.clip(counts / counts.sum(), 1e-8, None)
    weights   = (1.0 / density)
    weights   = weights / weights.mean()
    return torch.tensor(bin_edges, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)


def lookup_storm_weight(kp_batch, bin_edges, weights):
    bin_idx = torch.bucketize(kp_batch, bin_edges[1:-1])
    return weights.to(kp_batch.device)[bin_idx]


gl_weights      = make_gl_weights(CONFIG['H'], DEVICE)
sobolev_weights = make_sobolev_weights(CONFIG['l_max'], DEVICE)

_train_ds_kp = FalishaDTECDataset(CONFIG['data_root'], 'train', CONFIG['flip_latitude'])
_train_kp    = np.array(_train_ds_kp.omni_input[:, -1, CONFIG['kp_feature_idx']])
storm_bin_edges, storm_weights_table = build_storm_weight_table(_train_kp, CONFIG['n_kp_bins'])
del _train_ds_kp, _train_kp

print(f'GL weights: {gl_weights.shape} | Sobolev weights: {sobolev_weights.shape}')
print(f'Storm weight table: {CONFIG["n_kp_bins"]} bins, range '
      f'[{storm_weights_table.min():.2f}, {storm_weights_table.max():.2f}]')

GL weights: torch.Size([1, 1, 23, 1]) | Sobolev weights: torch.Size([1, 1, 23, 1])
Storm weight table: 18 bins, range [0.01, 7.08]


## Cell 6 — Model: SFNOBlock, SphericalConvGRUCell, TECSFNOModel

Gibbs window (Deliverable 3): `sigma_l = 1 - l/l_max`, applied after filter multiply, before inverse SHT.

In [ ]:
class SFNOBlock(nn.Module):
    def __init__(self, in_channels, out_channels, config):
        super().__init__()
        H, W, l_max = config['H'], config['W'], config['l_max']
        self.sht  = RealSHT(H, W, lmax=l_max, mmax=l_max, grid='legendre-gauss')
        self.isht = InverseRealSHT(H, W, lmax=l_max, mmax=l_max, grid='legendre-gauss')

        self.filter_re = nn.Parameter(torch.randn(out_channels, in_channels, l_max, l_max) * 0.02)
        self.filter_im = nn.Parameter(torch.randn(out_channels, in_channels, l_max, l_max) * 0.02)

        l = torch.arange(l_max, dtype=torch.float32)
        self.register_buffer('gibbs_sigma', (1 - l / l_max).view(1, 1, l_max, 1))

        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.act       = nn.GELU()

    def forward(self, x):
        residual   = self.pointwise(x)
        x_sht      = self.sht(x)
        w          = torch.complex(self.filter_re, self.filter_im)
        x_filtered = torch.einsum('bilm,oilm->bolm', x_sht, w) * self.gibbs_sigma
        return self.act(self.isht(x_filtered) + residual)


class SphericalConvGRUCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, config):
        super().__init__()
        combined = input_channels + hidden_channels
        self.reset_gate  = SFNOBlock(combined, hidden_channels, config)
        self.update_gate = SFNOBlock(combined, hidden_channels, config)
        self.candidate   = SFNOBlock(combined, hidden_channels, config)

    def forward(self, x_t, h_prev):
        cat_xh  = torch.cat([x_t, h_prev], dim=1)
        r       = torch.sigmoid(self.reset_gate(cat_xh))
        z       = torch.sigmoid(self.update_gate(cat_xh))
        cat_xrh = torch.cat([x_t, r * h_prev], dim=1)
        h_cand  = torch.tanh(self.candidate(cat_xrh))
        return (1 - z) * h_prev + z * h_cand


class TECSFNOModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        H, W     = config['H'], config['W']
        channels = config['sfno_channels']
        self.channels = channels

        self.omni_gru     = nn.GRU(config['n_omni_features'], config['gru_hidden_size'], batch_first=True)
        self.context_proj = nn.Linear(config['gru_hidden_size'], channels[0])
        self.conv_gru      = SphericalConvGRUCell(1, channels[0], config)

        ins  = [channels[0], channels[1], channels[2], channels[3]]
        outs = [channels[1], channels[2], channels[3], channels[3]]
        self.sfno_blocks = nn.ModuleList([
            SFNOBlock(ins[i], outs[i], config) for i in range(config['n_sfno_blocks'])
        ])
        self.output_head = nn.Conv2d(channels[-1], config['n_horizons'], kernel_size=1)

    def forward(self, tec_input, omni_input):
        B, T, H, W = tec_input.shape
        _, h_n  = self.omni_gru(omni_input)
        h_init  = self.context_proj(h_n.squeeze(0))
        h = h_init.unsqueeze(-1).unsqueeze(-1).expand(B, self.channels[0], H, W).contiguous()

        for t in range(T):
            h = self.conv_gru(tec_input[:, t].unsqueeze(1), h)

        x = h
        for block in self.sfno_blocks:
            x = block(x)
        return self.output_head(x)


_model = TECSFNOModel(CONFIG).to(DEVICE)
print(f'TECSFNOModel parameters: {sum(p.numel() for p in _model.parameters() if p.requires_grad):,}')
del _model

TECSFNOModel parameters: 52,316,547


## Cell 7 — Loss, metrics, teacher-forcing schedule

`lambda_sobolev = 1e-7`, not Saksham's originally suggested `1e-4` -- at `1e-4` the raw Sobolev sum dominated the loss (~276 vs ~0.09 real MSE). Metrics use `CONFIG['horizon_names']` (t+2h/t+4h/t+6h now), not hardcoded t+1h/t+2h/t+3h.

In [ ]:
def sobolev_penalty(model, sobolev_weights):
    penalty = 0.0
    for block in model.sfno_blocks:
        penalty = penalty + (block.filter_re ** 2 * sobolev_weights.squeeze()).sum()
        penalty = penalty + (block.filter_im ** 2 * sobolev_weights.squeeze()).sum()
    return penalty


def tec_loss(pred, target, gl_weights, model, sobolev_weights, storm_weights, lambda_sobolev):
    sq_err         = (pred - target) ** 2 * gl_weights
    mse_per_sample = sq_err.mean(dim=[1, 2, 3])
    w              = storm_weights / storm_weights.sum()
    mse            = (w * mse_per_sample).sum()
    sob            = sobolev_penalty(model, sobolev_weights)
    return mse + lambda_sobolev * sob, mse.item(), sob.item()


def compute_metrics(pred, target, gl_weights, horizon_names):
    sq_err = (pred - target) ** 2 * gl_weights
    return {f'rmse_{name}': sq_err[:, i].mean().sqrt().item() for i, name in enumerate(horizon_names)}


def teacher_forcing_prob(epoch, total_epochs):
    return max(0.0, 1.0 - (epoch / total_epochs))

## Cell 8 — Train / validate

In [ ]:
def train_one_epoch(model, loader, optimizer, gl_weights, sobolev_weights,
                     storm_bin_edges, storm_weights_table, epoch, config, device):
    model.train()
    total_loss = 0.0
    n_batches = len(loader)

    for i, batch in enumerate(loader):
        tec_input  = batch['tec_input'].to(device)
        omni_input = batch['omni_input'].to(device)
        target     = batch['target'].to(device)
        kp         = omni_input[:, -1, config['kp_feature_idx']].contiguous()
        sw         = lookup_storm_weight(kp, storm_bin_edges.to(device), storm_weights_table.to(device))

        optimizer.zero_grad()
        pred = model(tec_input, omni_input)
        loss, _, _ = tec_loss(pred, target, gl_weights, model, sobolev_weights, sw, config['lambda_sobolev'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        if i % 100 == 0:
            print(f'  batch {i}/{n_batches} | loss={loss.item():.4f}')

    return {'train/loss': total_loss / n_batches, 'train/teacher_prob': teacher_forcing_prob(epoch, config['n_epochs'])}


@torch.no_grad()
def validate_model(model, loader, gl_weights, sobolev_weights, storm_bin_edges,
                    storm_weights_table, config, device):
    model.eval()
    total_loss = 0.0
    rmse_accum = {name: 0.0 for name in config['horizon_names']}
    n = len(loader)

    for batch in loader:
        tec_input  = batch['tec_input'].to(device)
        omni_input = batch['omni_input'].to(device)
        target     = batch['target'].to(device)
        kp         = omni_input[:, -1, config['kp_feature_idx']].contiguous()
        sw         = lookup_storm_weight(kp, storm_bin_edges.to(device), storm_weights_table.to(device))

        pred = model(tec_input, omni_input)
        loss, _, _ = tec_loss(pred, target, gl_weights, model, sobolev_weights, sw, config['lambda_sobolev'])
        total_loss += loss.item()

        m = compute_metrics(pred, target, gl_weights, config['horizon_names'])
        for name in rmse_accum:
            rmse_accum[name] += m[f'rmse_{name}']

    result = {'val/loss': total_loss / n}
    result.update({f'rmse_{name}': rmse_accum[name] / n for name in rmse_accum})
    return result

## Cell 9 — Checkpointing

In [ ]:
def save_checkpoint(model, optimizer, epoch, val_loss, path='best_model.pt'):
    torch.save({'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'val_loss': val_loss, 'config': CONFIG}, path)
    print(f'  ✓ Saved checkpoint (epoch {epoch}, val_loss={val_loss:.4f})')


def load_checkpoint(model, optimizer, path='best_model.pt'):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f'  ✓ Loaded checkpoint from epoch {ckpt["epoch"]}')
    return ckpt['epoch'], ckpt['val_loss']

## Cell 10 — Main: run everything

In [ ]:
train_loader = make_dataloader(CONFIG, split='train')
val_loader   = make_dataloader(CONFIG, split='val')

model     = TECSFNOModel(CONFIG).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print('=' * 55)
print('TEC SFNO -- Residual, Reformatted Data')
print('=' * 55)
print(f"Grid:     {CONFIG['H']}x{CONFIG['W']} Gauss-Legendre")
print(f"Horizons: {CONFIG['horizon_names']}")
print(f"Device:   {DEVICE}")
print(f"Params:   {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

wandb.init(project=CONFIG['wandb_project'], name=CONFIG['wandb_run_name'], config=CONFIG)

best_val_loss = float('inf')
print('\nTraining...')

for epoch in range(CONFIG['n_epochs']):
    train_m = train_one_epoch(model, train_loader, optimizer, gl_weights, sobolev_weights,
                               storm_bin_edges, storm_weights_table, epoch, CONFIG, DEVICE)
    val_m = validate_model(model, val_loader, gl_weights, sobolev_weights,
                            storm_bin_edges, storm_weights_table, CONFIG, DEVICE)
    scheduler.step(val_m['val/loss'])

    wandb.log({**train_m, **val_m, 'epoch': epoch, 'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    rmse_str = ' | '.join(f"rmse_{name}={val_m[f'rmse_{name}']:.4f}" for name in CONFIG['horizon_names'])
    print(f"Epoch {epoch:03d} | train={train_m['train/loss']:.4f} | val={val_m['val/loss']:.4f} | "
          f"{rmse_str} | tf={train_m['train/teacher_prob']:.2f}")

    if val_m['val/loss'] < best_val_loss:
        best_val_loss = val_m['val/loss']
        save_checkpoint(model, optimizer, epoch, best_val_loss)
        wandb.save('best_model.pt')
        shutil.copy('best_model.pt', '/content/drive/MyDrive/tec_data/best_model.pt')
        print('  -> backed up to Drive')

wandb.finish()
print('\nDone.')

TEC SFNO -- Residual, Reformatted Data
Grid:     23x45 Gauss-Legendre
Horizons: ['t+2h', 't+4h', 't+6h']
Device:   cuda
Params:   52,316,547



Training...
  batch 0/1330 | loss=0.3284
  batch 100/1330 | loss=0.1806
  batch 200/1330 | loss=0.0839
  batch 300/1330 | loss=0.0750
  batch 400/1330 | loss=0.0397
  batch 500/1330 | loss=0.0276
  batch 600/1330 | loss=0.0236
  batch 700/1330 | loss=0.0251
  batch 800/1330 | loss=0.0162
  batch 900/1330 | loss=0.0118
  batch 1000/1330 | loss=0.0261
  batch 1100/1330 | loss=0.0187
  batch 1200/1330 | loss=0.0175
  batch 1300/1330 | loss=0.0145
Epoch 000 | train=0.0554 | val=0.0089 | rmse_t+2h=0.0640 | rmse_t+4h=0.0868 | rmse_t+6h=0.0978 | tf=1.00


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  ✓ Saved checkpoint (epoch 0, val_loss=0.0089)
  -> backed up to Drive
  batch 0/1330 | loss=0.0180
  batch 100/1330 | loss=0.0231
  batch 200/1330 | loss=0.0126
  batch 300/1330 | loss=0.0210
  batch 400/1330 | loss=0.0137
  batch 500/1330 | loss=0.0121
  batch 600/1330 | loss=0.0161
  batch 700/1330 | loss=0.0146
  batch 800/1330 | loss=0.0402
  batch 900/1330 | loss=0.0112
  batch 1000/1330 | loss=0.0236
  batch 1100/1330 | loss=0.0215
  batch 1200/1330 | loss=0.0126
  batch 1300/1330 | loss=0.0136
Epoch 001 | train=0.0199 | val=0.0080 | rmse_t+2h=0.0564 | rmse_t+4h=0.0815 | rmse_t+6h=0.0939 | tf=0.95
  ✓ Saved checkpoint (epoch 1, val_loss=0.0080)
  -> backed up to Drive
  batch 0/1330 | loss=0.0132
  batch 100/1330 | loss=0.0194
  batch 200/1330 | loss=0.0162
  batch 300/1330 | loss=0.0271
  batch 400/1330 | loss=0.0163
  batch 500/1330 | loss=0.0182
  batch 600/1330 | loss=0.0122
  batch 700/1330 | loss=0.0154
  batch 800/1330 | loss=0.0123
  batch 900/1330 | loss=0.0128
  batch

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
rmse_t+2h,█▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▂
rmse_t+4h,█▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▂
rmse_t+6h,█▇▇▆▅▅▄▄▃▃▃▃▂▂▂▂▁▁▁▂
train/loss,█▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train/teacher_prob,██▇▇▇▆▆▅▅▅▄▄▄▃▃▂▂▂▁▁
val/loss,█▆▆▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▂
epoch,19
lr,0.0001
rmse_t+2h,0.05026



Done.
